# Building a Custom Tokenizer From Scratch

Notebook 01 compared three **already-trained** tokenizers. This one trains one **yourself**, end to end — small enough to actually watch happen, and structured so you reproduce `custom-gpt-6m`'s real Hindi bug from Notebook 01 in miniature, on purpose, before fixing it the same way `custom-gpt-350m` actually did.

Every knob you touch here is a real knob in this repo's own tokenizer-training code — `custom-gpt-350m/src/gpt/tokenizer.py`'s `build_trainer_tokenizer()` and `custom-gpt-6m/src/gpt/data/prepare.py`'s `train_tokenizer()` call the exact same library functions, just with different arguments. This notebook is "read those two functions with your hands on the keyboard," not new material.

As in Notebook 01: predict before you run.

In [ ]:
## Setup (boilerplate)

from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

# A tiny, deliberately English-only, deliberately repetitive corpus — small enough that
# a 300-token vocab already shows real merges, and narrow enough to make Exercise 1's
# out-of-corpus failure reproducible on purpose.
corpus = [
    "the rabbit ran through the forest",
    "the little rabbit liked the forest",
    "a bird flew over the forest",
    "the bird and the rabbit were friends",
    "they played in the forest every day",
    "the forest was full of tall trees",
    "the little bird sang a happy song",
    "the rabbit and the bird went home",
] * 20  # repeated so BPE has real frequency signal to work with

print(f"{len(corpus)} lines, {sum(len(l) for l in corpus):,} characters total")

## Exercise 1 — Train a tokenizer, then break it on purpose

Below is a training call with **no `initial_alphabet`** — exactly `custom-gpt-6m`'s actual configuration, the one Notebook 01 proved fails on unseen scripts. `min_frequency=2` means a pair must appear at least twice to be eligible for merging.

**Predict first**: after training on the tiny English-only corpus above, what happens if you encode a Hindi word afterward — same failure as `custom-gpt-6m`, or different, given this corpus is even smaller and narrower? Run it and check.

In [ ]:
# No initial_alphabet — matches custom-gpt-6m/src/gpt/data/prepare.py's train_tokenizer() exactly
tok_broken = Tokenizer(models.BPE(unk_token="<unk>"))
tok_broken.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok_broken.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=300,
    special_tokens=["<unk>", "<|endoftext|>"],
    min_frequency=2,
)
tok_broken.train_from_iterator(corpus, trainer=trainer)
print("actual vocab size:", tok_broken.get_vocab_size())

# TODO: encode a Hindi word (or any text using characters that never appeared above)
# and check tok_broken.decode(...) against the original — does it round-trip?
test_text = "नमस्ते"
ids = tok_broken.encode(test_text).ids
print("tokens:", [tok_broken.id_to_token(i) for i in ids])
print("decoded:", repr(tok_broken.decode(ids)))
print("round-trips:", tok_broken.decode(ids) == test_text)

## Exercise 2 — Fix it the way `custom-gpt-350m` actually does

`custom-gpt-350m/src/gpt/tokenizer.py` passes one extra argument to `BpeTrainer` that the broken version above didn't: `initial_alphabet=pre_tokenizers.ByteLevel.alphabet()`. This forces all 256 possible byte-values into the vocabulary from the start, regardless of what appeared in training.

**Predict first**: will this cost you meaningfully more vocabulary space out of your 300-token budget? (Think about it before running — 256 is a large fraction of 300.) Retrain with the fix below and re-run the same Hindi round-trip check.

In [ ]:
tok_fixed = Tokenizer(models.BPE(unk_token="<unk>"))
tok_fixed.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok_fixed.decoder = decoders.ByteLevel()

trainer_fixed = trainers.BpeTrainer(
    vocab_size=300,
    special_tokens=["<unk>", "<|endoftext|>"],
    min_frequency=2,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),  # <-- the one-line fix
)
tok_fixed.train_from_iterator(corpus, trainer=trainer_fixed)
print("actual vocab size:", tok_fixed.get_vocab_size())

ids = tok_fixed.encode(test_text).ids
print("tokens:", [tok_fixed.id_to_token(i) for i in ids])
print("decoded:", repr(tok_fixed.decode(ids)))
print("round-trips:", tok_fixed.decode(ids) == test_text)

# TODO: compare tok_broken.get_vocab_size() vs tok_fixed.get_vocab_size() —
# how much of the 300-token budget did forcing the full byte alphabet actually cost?
# Is that trade-off obviously worth it, or does it depend on the target vocab size?

## Exercise 3 — This is literally what you implemented by hand in Notebook 01

Notebook 01's Exercise 4 had you implement one BPE merge step manually on a toy corpus. `tok_fixed` just ran that exact process thousands of times, in order, on the corpus above. Its learned merge list is inspectable — save the tokenizer and look at the raw JSON.

**Predict first**: given the corpus is dominated by "the", "forest", "rabbit", and "bird", what do you expect the *first* few learned merges to be — single-character pairs building toward common short words, or something else? Check below.

In [ ]:
import json
from pathlib import Path

out_path = Path("tiny_custom_tokenizer.json")
tok_fixed.save(str(out_path))

data = json.loads(out_path.read_text())
print(f"{len(data['model']['merges'])} merges learned. First 15, in the exact order BPE learned them:")
for m in data["model"]["merges"][:15]:
    print(" ", m)

# TODO: find where "forest", "rabbit", or "bird" first appears as a fully-merged single
# token in the merge list (not just its component pairs) — how many merges did it take?

## Wrap-up

`tiny_custom_tokenizer.json` (written above) is a real, valid tokenizer file — the exact same format as `custom-gpt-6m/data/tokenizer.json` and `custom-gpt-350m/tokenizer/tokenizer.json`, just trained on 8 lines instead of tens of thousands. Everything that works on those works on this one too: `Tokenizer.from_file()`, encode/decode, and — Notebook 03 — wrapping it in `transformers.PreTrainedTokenizerFast`.

Delete `tiny_custom_tokenizer.json` when you're done if you don't want it sitting in version control — it's a throwaway artifact from this notebook, not something worth keeping around.